## Making healpix cubes from (unconvolved) DRAGONS data for Y. Uchibori's project
## A. Ordog, April 2025

In [ ]:
import numpy as np
from astropy.io import fits as fits
from astropy.wcs import WCS
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
from astropy.coordinates import ICRS, Galactic, FK4, FK5
from scipy.optimize import curve_fit
from reproject import reproject_from_healpix, reproject_to_healpix
import healpy as hp
import seaborn as sb
from pygdsm import GlobalSkyModel
from scipy import interpolate
import astropy.units as u
from scipy.stats import linregress

In [ ]:
def read_in_cubes():

    cubes = {'E':{},
             'W':{},
             'C':{}}
    
    hdu = fits.open('/srv/data/dva/dragons_hoard/Data_cubes_Dec_2024_BB/Ecubes/DRAGONS_350_1030_binned_East_q_cube.fits')
    cubes['E']['q'] = hdu[0].data
    hdu = fits.open('/srv/data/dva/dragons_hoard/Data_cubes_Dec_2024_BB/Ecubes/DRAGONS_350_1030_binned_East_u_cube.fits')
    cubes['E']['u'] = hdu[0].data

    hdu = fits.open('/srv/data/dva/dragons_hoard/Data_cubes_Dec_2024_BB/Wcubes/DRAGONS_350_1030_binned_West_q_cube.fits')
    cubes['W']['q'] = hdu[0].data
    hdu = fits.open('/srv/data/dva/dragons_hoard/Data_cubes_Dec_2024_BB/Wcubes/DRAGONS_350_1030_binned_West_u_cube.fits')
    cubes['W']['u'] = hdu[0].data

    hdu = fits.open('/srv/data/dva/dragons_hoard/Data_cubes_Dec_2024_BB/merged_cubes/DRAGONS_350_1030_binned_q_cube.fits')
    cubes['C']['q'] = hdu[0].data
    hdu = fits.open('/srv/data/dva/dragons_hoard/Data_cubes_Dec_2024_BB/merged_cubes/DRAGONS_350_1030_binned_u_cube.fits')
    cubes['C']['u'] = hdu[0].data
    hdu = fits.open('/srv/data/dva/dragons_hoard/Data_cubes_Dec_2024_BB/merged_cubes/DRAGONS_350_1030_binned_i_cube.fits')
    cubes['C']['i'] = hdu[0].data

    hdr = hdu[0].header

    freqs = WCS(hdr).all_pix2world(0,0,range(cubes['C']['q'].shape[0]),0)[2]
    dec   = WCS(hdr).all_pix2world(0,range(cubes['C']['q'].shape[1]),0,0)[1]
    ra    = WCS(hdr).all_pix2world(range(cubes['C']['q'].shape[2]),0,0,0)[0]

    for EW in ['E', 'W', 'C']:
        for stokes in ['q', 'u']:
            cubes[EW][stokes][cubes[EW][stokes] == 0.0] = np.nan
    cubes['C']['i'][cubes['C']['i'] == 0.0] = np.nan

    wcs2D = WCS(hdr).dropaxis(2)

    return cubes, hdr, wcs2D, freqs, ra, dec

### Read in DRAGONS data

In [ ]:
cubes, hdr, wcs2D, freqs, ra, dec = read_in_cubes()
print(cubes['C']['i'].shape)


### Calculate DRAGONS error from East and West

In [ ]:
dq = np.abs(cubes['E']['q'] - cubes['W']['q'])/2
du = np.abs(cubes['E']['u'] - cubes['W']['u'])/2


### Convert DRAGONS I, Q, U and error cube to Healpix

In [ ]:
nside = 64

dq_hpx  = np.empty([len(freqs),hp.nside2npix(nside)])
du_hpx  = np.empty([len(freqs),hp.nside2npix(nside)])
q_hpx   = np.empty([len(freqs),hp.nside2npix(nside)])
u_hpx   = np.empty([len(freqs),hp.nside2npix(nside)])
i_hpx   = np.empty([len(freqs),hp.nside2npix(nside)])

for i in range(0,len(freqs)): 
    print(freqs[i]/1e6)
    q_hpx[i],  footprint = reproject_to_healpix((cubes['C']['q'][i], wcs2D.to_header()), 'icrs', nside=nside)
    u_hpx[i],  footprint = reproject_to_healpix((cubes['C']['u'][i], wcs2D.to_header()), 'icrs', nside=nside)
    i_hpx[i],  footprint = reproject_to_healpix((cubes['C']['i'][i], wcs2D.to_header()), 'icrs', nside=nside)
    dq_hpx[i],footprint = reproject_to_healpix((dq[i], wcs2D.to_header()), 'icrs', nside=nside)
    du_hpx[i],footprint = reproject_to_healpix((du[i], wcs2D.to_header()), 'icrs', nside=nside)
    

### Convert DRAGONS to K

In [ ]:
JytoK_all = np.load('/srv/data/dva/conversion_factor_JytoK_July2024_BB.npy')
print(JytoK_all[0].shape)

f = interpolate.interp1d(JytoK_all[0], JytoK_all[1])
JytoK = f(freqs/1e6)

fig, ax = plt.subplots(1,1,figsize=(10,2))
ax.scatter(JytoK_all[0],JytoK_all[1], s=1)
ax.scatter(freqs/1e6, JytoK, s=1)

dq_hpx_K = dq_hpx*np.expand_dims(JytoK,axis=1)
du_hpx_K = du_hpx*np.expand_dims(JytoK,axis=1)
q_hpx_K  = q_hpx*np.expand_dims(JytoK,axis=1)
u_hpx_K  = u_hpx*np.expand_dims(JytoK,axis=1)
i_hpx_K  = i_hpx*np.expand_dims(JytoK,axis=1)

### Save all healpix files so far

In [ ]:
hdr_hpx = fits.Header()
hdr_hpx['SIMPLE'] = True
hdr_hpx['BITPIX'] = -32 
hdr_hpx['NAXIS'] = 2
hdr_hpx['NAXIS1'] = hp.nside2npix(64)  
hdr_hpx['NAXIS2'] = len(freqs)  
hdr_hpx['PIXTYPE'] = 'HEALPIX'  
hdr_hpx['ORDERING'] = 'RING'
hdr_hpx['INDXSCHM'] = 'IMPLICIT'           
hdr_hpx['OBJECT']   = 'FULLSKY '       
hdr_hpx['COORDSYS'] = 'C       '   
hdr_hpx['NSIDE'] = 64       
hdr_hpx['CRVAL2'] = freqs[0]/1e6       
hdr_hpx['CDELT2'] = (freqs[1]-freqs[0])/1e6        
hdr_hpx['CRPIX2'] = 1.0        
hdr_hpx['CTYPE2'] = 'FREQ'     
hdr_hpx['CUNIT2'] = 'MHz'

print(repr(hdr_hpx))

In [ ]:
fits.writeto('/home/aordog/Dropbox/DRAGONS_for_CHIME/Q_eq_K.hpx.fits', q_hpx_K, header=hdr_hpx, overwrite=True)
fits.writeto('/home/aordog/Dropbox/DRAGONS_for_CHIME/U_eq_K.hpx.fits', u_hpx_K, header=hdr_hpx, overwrite=True)
fits.writeto('/home/aordog/Dropbox/DRAGONS_for_CHIME/I_eq_K.hpx.fits', i_hpx_K, header=hdr_hpx, overwrite=True)
fits.writeto('/home/aordog/Dropbox/DRAGONS_for_CHIME/dQ_eq_K.hpx.fits', dq_hpx_K, header=hdr_hpx, overwrite=True)
fits.writeto('/home/aordog/Dropbox/DRAGONS_for_CHIME/dU_eq_K.hpx.fits', du_hpx_K, header=hdr_hpx, overwrite=True)


## Fixing Stokes I...

### Read in saved PyGDSM file

In [ ]:
hdu = fits.open('/srv/data/dva/dragons_hoard/Data_cubes_errors_Mar2025/pygdsm_eq_K.hpx.fits')
pygdsm = hdu[0].data
print(pygdsm.shape)

### Make ra and dec arrays

In [ ]:
all_pix = np.arange(0,hp.nside2npix(hdr_hpx['NSIDE']),1)
dec, ra = np.degrees(hp.pix2ang(nside=hdr_hpx['NSIDE'], ipix=all_pix))
dec = 90-dec

### Make bright source mask

In [ ]:
def rem_source(src, r, mask, ra, dec):

    c = SkyCoord.from_name(src) 
    print(c.ra.deg,c.dec.deg)

    wbad = np.where(np.sqrt((ra-c.ra.deg)**2+(dec-c.dec.deg)**2) <= r)
    mask[wbad] = np.nan
    
    return mask

mask = np.ones(hp.nside2npix(hdr_hpx['NSIDE']))

mask = rem_source("CASA", 5, mask, ra, dec)
mask = rem_source("CYGA", 5, mask, ra, dec)
mask = rem_source("M1",   5, mask, ra, dec) # Tau A
mask = rem_source("M87",  5, mask, ra, dec) # Vir A

hp.cartview(mask)

### Calculate TT-plot slopes and intercepts and save them to arrays

In [ ]:
slopes     = np.empty_like(freqs)
intercepts = np.empty_like(freqs)
rvalues    = np.empty_like(freqs)

for fidx in range(0,len(freqs)):
    print(freqs[fidx]/1e6)
    try:
        idx_good = np.where(np.isfinite(pygdsm[fidx]*mask) & (np.isfinite(i_hpx_K[fidx]*mask)))[0]
        slopes[fidx], intercepts[fidx], rvalues[fidx], p, se = linregress(pygdsm[fidx][idx_good], i_hpx_K[fidx][idx_good])
    except:
        pass

slopes[intercepts >= 0]     = np.nan
rvalues[intercepts >= 0]    = np.nan
intercepts[intercepts >= 0] = np.nan


In [ ]:
fig, ax = plt.subplots(1,1,figsize=(14,5))

fs = 14
s = 10

ax.scatter(freqs/1e6, slopes, s=s, color='blue',label='slope')
ax.scatter(freqs/1e6, rvalues, s=s, color='C0',label='r-value')
ax.set_xlim(350,1030)
ax.set_ylim(0.5,1.5)
ax.grid(True)
ax.legend(loc='upper left',fontsize=fs, markerscale=2)
ax.set_ylabel('slope and r-value',fontsize=fs)
ax.set_xlabel('frequency (MHz)',fontsize=fs)

ax2 = ax.twinx()
ax2.scatter(freqs/1e6, intercepts, s=s, color='k',label='intercept')
ax2.scatter(freqs/1e6, np.nanmin(mask*i_hpx_K,axis=1), s=s, color='grey',label='DRAGONS min')
ax2.set_ylim(-40,10)
ax2.legend(loc='upper right',fontsize=fs, markerscale=2)
ax2.set_ylabel(r'y-intercept and DRAGONS min (K)',fontsize=fs)

for axs in [ax, ax2]:
    axs.tick_params(axis='both', labelsize=fs)

plt.savefig('/home/aordog/Dropbox/plots/dragons/errors/dragons_pygdsm_offsets_NOTconvolved.png')

### Apply offset correction and save new FITS file

In [ ]:
savenew = True
if savenew:
    StokesI_correct = i_hpx_K - np.expand_dims(intercepts,1)
    fits.writeto('/home/aordog/Dropbox/DRAGONS_for_CHIME/I_eq_K_offset_corr.hpx.fits', StokesI_correct, header=hdr_hpx, overwrite=True)


### Calculate Stokes I error and save

In [ ]:
dI = np.abs(StokesI_correct - pygdsm)/2
fits.writeto('/home/aordog/Dropbox/DRAGONS_for_CHIME/dI_eq_K.hpx.fits', dI, header=hdr_hpx, overwrite=True)

In [ ]:
beams = np.load('/srv/rbooth02/cygbeam.npy')
print(beams.shape)
beamfreqs = np.load('/srv/rbooth02/abscalfreqs.npy')
print(beamfreqs.shape)

f = interpolate.interp1d(beamfreqs, 2.355*beams)
beamwidth = f(freqs/1e6)

fig, ax = plt.subplots(1,1,figsize=(10,5))
ax.scatter(beamfreqs, 2.355*beams, s=1)
ax.scatter(freqs/1e6, beamwidth, s=1)

In [ ]:
beams_out = np.stack([freqs/1e6, beamwidth], axis=1)

np.savetxt("/home/aordog/Dropbox/DRAGONS_for_CHIME/dragons_beam_widths.txt", beams_out, delimiter=" ")
